# Семинар 3. Kafka, CDC и Structured Streaming на PySpark

In [1]:
import os
import sys
import shutil
from pathlib import Path

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

java21_home = Path("/Library/Java/JavaVirtualMachines/jdk-21.jdk/Contents/Home")
if java21_home.exists():
    os.environ["JAVA_HOME"] = str(java21_home)

from pyspark.sql import SparkSession, functions as F, types as T, Window

spark = (
    SparkSession.builder
    .appName("etl-course-streaming-cdc-demo")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/05 17:01:21 WARN Utils: Your hostname, MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.11 instead (on interface en0)
26/05/05 17:01:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/05 17:01:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
import json
import shutil
from datetime import datetime, timedelta, timezone
from pathlib import Path


BASE_TIME = datetime(2026, 5, 5, 9, 0, tzinfo=timezone.utc)


def clean_dir(path: Path) -> None:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def event(
    *,
    order_id: int,
    user_id: int,
    op: str,
    status: str | None,
    amount: float | None,
    event_minutes: int,
    ingest_minutes: int,
    event_id: str,
    city: str,
) -> dict:
    return {
        "event_id": event_id,
        "order_id": order_id,
        "user_id": user_id,
        "op": op,
        "status": status,
        "amount": amount,
        "city": city,
        "event_ts": (BASE_TIME + timedelta(minutes=event_minutes)).isoformat(),
        "ingest_ts": (BASE_TIME + timedelta(minutes=ingest_minutes)).isoformat(),
    }


def build_batches():
    return [
        [
            event(order_id=101, user_id=1, op="c", status="created", amount=1200.0, event_minutes=1, ingest_minutes=2, event_id="e001", city="Moscow"),
            event(order_id=102, user_id=2, op="c", status="created", amount=850.0, event_minutes=2, ingest_minutes=3, event_id="e002", city="Kazan"),
            event(order_id=103, user_id=3, op="c", status="created", amount=420.0, event_minutes=3, ingest_minutes=4, event_id="e003", city="Moscow"),
        ],
        [
            event(order_id=101, user_id=1, op="u", status="paid", amount=1200.0, event_minutes=8, ingest_minutes=9, event_id="e004", city="Moscow"),
            event(order_id=102, user_id=2, op="u", status="cancelled", amount=850.0, event_minutes=9, ingest_minutes=10, event_id="e005", city="Kazan"),
            event(order_id=104, user_id=4, op="c", status="created", amount=2200.0, event_minutes=10, ingest_minutes=11, event_id="e006", city="Sochi"),
        ],
        [
            event(order_id=105, user_id=5, op="c", status="created", amount=640.0, event_minutes=12, ingest_minutes=13, event_id="e007", city="Perm"),
            event(order_id=103, user_id=3, op="u", status="paid", amount=420.0, event_minutes=14, ingest_minutes=15, event_id="e008", city="Moscow"),
            event(order_id=104, user_id=4, op="u", status="paid", amount=2200.0, event_minutes=16, ingest_minutes=17, event_id="e009", city="Sochi"),
        ],
        [
            event(order_id=102, user_id=2, op="u", status="paid", amount=850.0, event_minutes=6, ingest_minutes=25, event_id="e010_late_old", city="Kazan"),
            event(order_id=105, user_id=5, op="d", status=None, amount=None, event_minutes=26, ingest_minutes=27, event_id="e011", city="Perm"),
            event(order_id=106, user_id=6, op="c", status="created", amount=310.0, event_minutes=28, ingest_minutes=29, event_id="e012", city="Moscow"),
        ],
    ]


def write_batch(source_dir: Path, batch_id: int, rows: list[dict]) -> Path:
    source_dir.mkdir(parents=True, exist_ok=True)
    path = source_dir / f"batch_{batch_id:02d}.jsonl"
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    return path


def write_batches(source_dir: Path, batches: list[list[dict]], count: int | None = None) -> list[Path]:
    selected = batches if count is None else batches[:count]
    return [write_batch(source_dir, idx + 1, rows) for idx, rows in enumerate(selected)]


def event_schema_ddl() -> str:
    return """
        event_id STRING,
        order_id INT,
        user_id INT,
        op STRING,
        status STRING,
        amount DOUBLE,
        city STRING,
        event_ts STRING,
        ingest_ts STRING
    """

In [3]:
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    PROJECT_ROOT = ROOT.parent
else:
    PROJECT_ROOT = ROOT

LAB_DIR = PROJECT_ROOT / "sample_projects" / "07_streaming_cdc"
sys.path.insert(0, str(LAB_DIR))


OUT = PROJECT_ROOT / "notebooks" / "streaming_cdc_output"
SOURCE = OUT / "source_topic_orders"
BRONZE = OUT / "bronze_orders_events"
CHECKPOINT = OUT / "checkpoint_orders_to_bronze"
GOLD = OUT / "gold"

clean_dir(OUT)
SOURCE.mkdir(parents=True, exist_ok=True)
batches = build_batches()
len(batches), sum(len(batch) for batch in batches)

(4, 12)

In [4]:
batches[0]

[{'event_id': 'e001',
  'order_id': 101,
  'user_id': 1,
  'op': 'c',
  'status': 'created',
  'amount': 1200.0,
  'city': 'Moscow',
  'event_ts': '2026-05-05T09:01:00+00:00',
  'ingest_ts': '2026-05-05T09:02:00+00:00'},
 {'event_id': 'e002',
  'order_id': 102,
  'user_id': 2,
  'op': 'c',
  'status': 'created',
  'amount': 850.0,
  'city': 'Kazan',
  'event_ts': '2026-05-05T09:02:00+00:00',
  'ingest_ts': '2026-05-05T09:03:00+00:00'},
 {'event_id': 'e003',
  'order_id': 103,
  'user_id': 3,
  'op': 'c',
  'status': 'created',
  'amount': 420.0,
  'city': 'Moscow',
  'event_ts': '2026-05-05T09:03:00+00:00',
  'ingest_ts': '2026-05-05T09:04:00+00:00'}]

## 1. События: что будет приходить в поток

В CDC каждое событие описывает изменение бизнес-сущности.  
В нашем примере сущность - заказ, ключ - `order_id`, а операция лежит в поле `op`.

In [5]:
schema = T.StructType([
    T.StructField("event_id", T.StringType(), False),
    T.StructField("order_id", T.IntegerType(), False),
    T.StructField("user_id", T.IntegerType(), False),
    T.StructField("op", T.StringType(), False),
    T.StructField("status", T.StringType(), True),
    T.StructField("amount", T.DoubleType(), True),
    T.StructField("city", T.StringType(), True),
    T.StructField("event_ts", T.StringType(), False),
    T.StructField("ingest_ts", T.StringType(), False),
])

preview = spark.createDataFrame([row for batch in batches for row in batch], schema=schema)
preview.orderBy("ingest_ts", "event_id").show(truncate=False)

+-------------+--------+-------+---+---------+------+------+-------------------------+-------------------------+
|event_id     |order_id|user_id|op |status   |amount|city  |event_ts                 |ingest_ts                |
+-------------+--------+-------+---+---------+------+------+-------------------------+-------------------------+
|e001         |101     |1      |c  |created  |1200.0|Moscow|2026-05-05T09:01:00+00:00|2026-05-05T09:02:00+00:00|
|e002         |102     |2      |c  |created  |850.0 |Kazan |2026-05-05T09:02:00+00:00|2026-05-05T09:03:00+00:00|
|e003         |103     |3      |c  |created  |420.0 |Moscow|2026-05-05T09:03:00+00:00|2026-05-05T09:04:00+00:00|
|e004         |101     |1      |u  |paid     |1200.0|Moscow|2026-05-05T09:08:00+00:00|2026-05-05T09:09:00+00:00|
|e005         |102     |2      |u  |cancelled|850.0 |Kazan |2026-05-05T09:09:00+00:00|2026-05-05T09:10:00+00:00|
|e006         |104     |4      |c  |created  |2200.0|Sochi |2026-05-05T09:10:00+00:00|2026-05-05

## 2. Первый микробатч: пишем события в учебный topic

In [6]:
def write_batch(source_dir: Path, batch_id: int, rows: list[dict]) -> Path:
    source_dir.mkdir(parents=True, exist_ok=True)
    path = source_dir / f"batch_{batch_id:02d}.jsonl"
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    return path

In [7]:
first_file = write_batch(SOURCE, 1, batches[0])
print(first_file)
print(first_file.read_text(encoding="utf-8"))

/Users/avmysh/projs/2026/etl/notebooks/streaming_cdc_output/source_topic_orders/batch_01.jsonl
{"event_id": "e001", "order_id": 101, "user_id": 1, "op": "c", "status": "created", "amount": 1200.0, "city": "Moscow", "event_ts": "2026-05-05T09:01:00+00:00", "ingest_ts": "2026-05-05T09:02:00+00:00"}
{"event_id": "e002", "order_id": 102, "user_id": 2, "op": "c", "status": "created", "amount": 850.0, "city": "Kazan", "event_ts": "2026-05-05T09:02:00+00:00", "ingest_ts": "2026-05-05T09:03:00+00:00"}
{"event_id": "e003", "order_id": 103, "user_id": 3, "op": "c", "status": "created", "amount": 420.0, "city": "Moscow", "event_ts": "2026-05-05T09:03:00+00:00", "ingest_ts": "2026-05-05T09:04:00+00:00"}



## 3. Streaming read: из topic в Bronze

In [8]:
def run_bronze_stream():
    stream_df = (
        spark.readStream
        .schema(schema)
        .json(str(SOURCE))
        .withColumn("event_ts", F.to_timestamp("event_ts"))
        .withColumn("ingest_ts", F.to_timestamp("ingest_ts"))
        .withColumn("bronze_loaded_at", F.current_timestamp())
    )

    query = (
        stream_df.writeStream
        .format("parquet")
        .option("path", str(BRONZE))
        .option("checkpointLocation", str(CHECKPOINT))
        .outputMode("append")
        .trigger(availableNow=True)
        .start()
    )
    query.awaitTermination()
    return query

spark.sparkContext.setJobDescription("01 stream first batch to bronze")
run_bronze_stream()

bronze = spark.read.parquet(str(BRONZE))
bronze.orderBy("ingest_ts", "event_id").show(truncate=False)

26/05/05 17:02:59 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


+--------+--------+-------+---+-------+------+------+-------------------+-------------------+----------------------+
|event_id|order_id|user_id|op |status |amount|city  |event_ts           |ingest_ts          |bronze_loaded_at      |
+--------+--------+-------+---+-------+------+------+-------------------+-------------------+----------------------+
|e001    |101     |1      |c  |created|1200.0|Moscow|2026-05-05 09:01:00|2026-05-05 09:02:00|2026-05-05 14:02:59.31|
|e002    |102     |2      |c  |created|850.0 |Kazan |2026-05-05 09:02:00|2026-05-05 09:03:00|2026-05-05 14:02:59.31|
|e003    |103     |3      |c  |created|420.0 |Moscow|2026-05-05 09:03:00|2026-05-05 09:04:00|2026-05-05 14:02:59.31|
+--------+--------+-------+---+-------+------+------+-------------------+-------------------+----------------------+



## 4. Checkpoint: повторный запуск не читает старые файлы

In [10]:
before = spark.read.parquet(str(BRONZE)).count()
before

3

In [14]:
spark.sparkContext.setJobDescription("02 rerun stream without new files")
run_bronze_stream()

after = spark.read.parquet(str(BRONZE)).count()
print(f"rows before={before}, rows after={after}")

rows before=3, rows after=3


26/05/05 17:05:49 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Файлы чекпоинта

In [15]:
for path in sorted(CHECKPOINT.rglob("*"))[:20]:
    print(path.relative_to(CHECKPOINT))

.metadata.crc
commits
commits/.0.crc
commits/0
metadata
offsets
offsets/.0.crc
offsets/0
sources
sources/0
sources/0/.0.crc
sources/0/0


## 5. Добавляем новые микробатчи

In [16]:
write_batch(SOURCE, 2, batches[1])
write_batch(SOURCE, 3, batches[2])

spark.sparkContext.setJobDescription("03 stream second and third batches")
run_bronze_stream()

bronze = spark.read.parquet(str(BRONZE))
print("Bronze rows:", bronze.count())
bronze.orderBy("ingest_ts", "event_id").show(truncate=False)

26/05/05 17:06:29 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Bronze rows: 9
+--------+--------+-------+---+---------+------+------+-------------------+-------------------+-----------------------+
|event_id|order_id|user_id|op |status   |amount|city  |event_ts           |ingest_ts          |bronze_loaded_at       |
+--------+--------+-------+---+---------+------+------+-------------------+-------------------+-----------------------+
|e001    |101     |1      |c  |created  |1200.0|Moscow|2026-05-05 09:01:00|2026-05-05 09:02:00|2026-05-05 14:02:59.31 |
|e002    |102     |2      |c  |created  |850.0 |Kazan |2026-05-05 09:02:00|2026-05-05 09:03:00|2026-05-05 14:02:59.31 |
|e003    |103     |3      |c  |created  |420.0 |Moscow|2026-05-05 09:03:00|2026-05-05 09:04:00|2026-05-05 14:02:59.31 |
|e004    |101     |1      |u  |paid     |1200.0|Moscow|2026-05-05 09:08:00|2026-05-05 09:09:00|2026-05-05 14:06:29.571|
|e005    |102     |2      |u  |cancelled|850.0 |Kazan |2026-05-05 09:09:00|2026-05-05 09:10:00|2026-05-05 14:06:29.571|
|e006    |104     |4     

## 6. Silver: собираем актуальное состояние заказов

Bronze хранит все события. Silver должен ответить на вопрос: какое состояние заказа актуально сейчас?

In [17]:
def build_current_orders(bronze_df):
    ranked = (
        bronze_df
        .dropDuplicates(["event_id"])
        .withColumn(
            "rn",
            F.row_number().over(
                Window.partitionBy("order_id")
                .orderBy(F.col("event_ts").desc(), F.col("ingest_ts").desc(), F.col("event_id").desc())
            )
        )
    )
    return (
        ranked
        .filter((F.col("rn") == 1) & (F.col("op") != "d"))
        .drop("rn", "bronze_loaded_at")
    )

current_orders = build_current_orders(bronze)
current_orders.orderBy("order_id").show(truncate=False)

+--------+--------+-------+---+---------+------+------+-------------------+-------------------+
|event_id|order_id|user_id|op |status   |amount|city  |event_ts           |ingest_ts          |
+--------+--------+-------+---+---------+------+------+-------------------+-------------------+
|e004    |101     |1      |u  |paid     |1200.0|Moscow|2026-05-05 09:08:00|2026-05-05 09:09:00|
|e005    |102     |2      |u  |cancelled|850.0 |Kazan |2026-05-05 09:09:00|2026-05-05 09:10:00|
|e008    |103     |3      |u  |paid     |420.0 |Moscow|2026-05-05 09:14:00|2026-05-05 09:15:00|
|e009    |104     |4      |u  |paid     |2200.0|Sochi |2026-05-05 09:16:00|2026-05-05 09:17:00|
|e007    |105     |5      |c  |created  |640.0 |Perm  |2026-05-05 09:12:00|2026-05-05 09:13:00|
+--------+--------+-------+---+---------+------+------+-------------------+-------------------+



## 7. Позднее событие: приехало сейчас, но случилось раньше

In [19]:
write_batch(SOURCE, 4, batches[3])

spark.sparkContext.setJobDescription("04 stream late event batch")
run_bronze_stream()

bronze = spark.read.parquet(str(BRONZE))
bronze.filter(F.col("order_id") == 102).orderBy("ingest_ts").show(truncate=False)

current_orders = build_current_orders(bronze)
current_orders.filter(F.col("order_id") == 102).show(truncate=False)

26/05/05 17:10:13 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


+-------------+--------+-------+---+---------+------+-----+-------------------+-------------------+-----------------------+
|event_id     |order_id|user_id|op |status   |amount|city |event_ts           |ingest_ts          |bronze_loaded_at       |
+-------------+--------+-------+---+---------+------+-----+-------------------+-------------------+-----------------------+
|e002         |102     |2      |c  |created  |850.0 |Kazan|2026-05-05 09:02:00|2026-05-05 09:03:00|2026-05-05 14:02:59.31 |
|e005         |102     |2      |u  |cancelled|850.0 |Kazan|2026-05-05 09:09:00|2026-05-05 09:10:00|2026-05-05 14:06:29.571|
|e010_late_old|102     |2      |u  |paid     |850.0 |Kazan|2026-05-05 09:06:00|2026-05-05 09:25:00|2026-05-05 14:10:13.178|
+-------------+--------+-------+---+---------+------+-----+-------------------+-------------------+-----------------------+

+--------+--------+-------+---+---------+------+-----+-------------------+-------------------+
|event_id|order_id|user_id|op |statu

## 8. Gold: свежая витрина по городам

In [20]:
gold_city = (
    current_orders
    .groupBy("city", "status")
    .agg(
        F.count("*").alias("orders_cnt"),
        F.round(F.sum("amount"), 2).alias("amount_sum"),
    )
    .orderBy("city", "status")
)

gold_city.show(truncate=False)

+------+---------+----------+----------+
|city  |status   |orders_cnt|amount_sum|
+------+---------+----------+----------+
|Kazan |cancelled|1         |850.0     |
|Moscow|created  |1         |310.0     |
|Moscow|paid     |2         |1620.0    |
|Sochi |paid     |1         |2200.0    |
+------+---------+----------+----------+



## 9. Окна и watermarks: идея на статическом примере

In [26]:
bronze.show()

+-------------+--------+-------+---+---------+------+------+-------------------+-------------------+--------------------+
|     event_id|order_id|user_id| op|   status|amount|  city|           event_ts|          ingest_ts|    bronze_loaded_at|
+-------------+--------+-------+---+---------+------+------+-------------------+-------------------+--------------------+
|         e001|     101|      1|  c|  created|1200.0|Moscow|2026-05-05 09:01:00|2026-05-05 09:02:00|2026-05-05 14:02:...|
|         e002|     102|      2|  c|  created| 850.0| Kazan|2026-05-05 09:02:00|2026-05-05 09:03:00|2026-05-05 14:02:...|
|         e003|     103|      3|  c|  created| 420.0|Moscow|2026-05-05 09:03:00|2026-05-05 09:04:00|2026-05-05 14:02:...|
|         e007|     105|      5|  c|  created| 640.0|  Perm|2026-05-05 09:12:00|2026-05-05 09:13:00|2026-05-05 14:06:...|
|         e008|     103|      3|  u|     paid| 420.0|Moscow|2026-05-05 09:14:00|2026-05-05 09:15:00|2026-05-05 14:06:...|
|         e009|     104|

In [25]:
bronze.count()

12

In [21]:
windowed = (
    bronze
    .filter(F.col("op") != "d")
    .groupBy(F.window("event_ts", "10 minutes").alias("event_window"))
    .agg(F.count("*").alias("events_cnt"), F.round(F.sum("amount"), 2).alias("amount_sum"))
    .select(
        F.col("event_window.start").alias("window_start"),
        F.col("event_window.end").alias("window_end"),
        "events_cnt",
        "amount_sum",
    )
    .orderBy("window_start")
)

windowed.show(truncate=False)

+-------------------+-------------------+----------+----------+
|window_start       |window_end         |events_cnt|amount_sum|
+-------------------+-------------------+----------+----------+
|2026-05-05 09:00:00|2026-05-05 09:10:00|6         |5370.0    |
|2026-05-05 09:10:00|2026-05-05 09:20:00|4         |5460.0    |
|2026-05-05 09:20:00|2026-05-05 09:30:00|1         |310.0     |
+-------------------+-------------------+----------+----------+



In [22]:
stream_df = (
    spark.readStream
    .schema(schema)
    .json(str(SOURCE))
    .withColumn("event_ts", F.to_timestamp("event_ts"))
    .withColumn("ingest_ts", F.to_timestamp("ingest_ts"))
    .withColumn("bronze_loaded_at", F.current_timestamp())
)

In [23]:
with_watermark = (
    stream_df
    .withWatermark("event_ts", "5 minutes")
    .filter(F.col("op") != "d")
    .groupBy(F.window("event_ts", "10 minutes").alias("w"))
    .agg(
        F.count("*").alias("events_cnt"),
        F.sum("amount").alias("amount_sum"),
    )
)

# Если мы хороший разработчик -- убиваем сессию спарка

In [28]:
spark.stop()